In [ ]:
%load_ext autoreload
%autoreload 2

import copy
from ipywidgets import interact, IntSlider, fixed, ToggleButton
from IPython.display import display
from edit import fm, cre, fd, ef, sea, pl, inter, gshota

# ファイルパスの設定
data_path = "data/outputData/sample/evaltest"

# ファイルマネージャー作成
manager = fm.FileManager(data_path)

# con.num_to_str変更

# データの読み込み
data_size = 0
data_list = manager.load_files("datas", 0, data_size)
ori_data_list = copy.deepcopy(data_list)
manual_data_list = copy.deepcopy(data_list)

In [ ]:
# 正解ファイル読み込み
label_data_list = manager.load_label_file()
label_data_list

In [ ]:
# フォーメーション情報の設定
formations_left, formations_right = manager.load_formation_file()
print(f"formations_left: {formations_left}")
print(f"formations_right: {formations_right}")
player_lists_left, player_lists_right = cre.create_player_lists(formations_left, formations_right)
print(f"player_lists_left: {player_lists_left}")
print(f"player_lists_right: {player_lists_right}")

# 補正用マネージャー作成
edit_func = ef.EditFunc(0, 0)
# games_0, games_1 = cre.create_games(formations_left, formations_right)
# print(f"games_0: {games_0}")
# print(f"games_1: {games_1}")

In [ ]:
# 手動補正のみ
edit_numbers = manager.load_edit_file()
edit_func.edit_data_manual(manual_data_list, edit_numbers)

In [ ]:
# 手動補正＋自動補正
data = data_list[0]

# 背番号削除
edit_func.delete_jersey_number(data_list)

# 手動補正
edit_numbers = manager.load_edit_file()
edit_func.edit_data_manual(data_list, edit_numbers)

# 自動補正
set_frames = {}
edit_func.edit_data_auto(data_list, player_lists_left, player_lists_right, 0.1, 20.0, 0.5, set_frames)

# 描画
try:
    last_frame_value
except NameError:
    last_frame_value = 1

def on_frame_change(change):
    global last_frame_value
    last_frame_value = change['new']

frame_slider = IntSlider(min=1, max=750, step=1, value=last_frame_value, description="Frame")
frame_slider.observe(on_frame_change, names='value')

plot_func = pl.Plot(data_path)
lims = [[-50.0, 50.0], [50.0, -50.0]]
colors = ["blue", "red"]
interact(
    plot_func.plot_frame_data,
    data_base=fixed(data_path),
    data=fixed(data),
    data_num=fixed(0),
    frame_num=frame_slider,
    lims=fixed(lims),
    colors=fixed(colors)
)

In [ ]:
# ID検索
sea.search_frame(data, 0, 29.0)
sea.search_track_id(data, 0, 484)

In [ ]:
# 画像を開く
picture_path = plot_func.display_picture(0, last_frame_value)
!code "{picture_path}"

In [ ]:
# 複数ある背番号の統合
inter.integrate_jerseis(data_list)

# 背番号被り確認
sea.search_multiple_frame(data_list, 0.0)

In [ ]:
# 正解座標描画
label_data = label_data_list[0]
interact(
    plot_func.plot_frame_data,
    data_base=fixed(data_path),
    data=fixed(label_data),
    data_num=fixed(0),
    frame_num=frame_slider,
    lims=fixed(lims),
    colors=fixed(colors)
)

In [ ]:
# GSHOTA計算
gshota_man = gshota.GSHOTA()
# 元データ
gs_hota, det_a, ass_a = gshota_man.calc_gs_hota(label_data_list[0], ori_data_list[0], 0)
print(f"元データ: GS-HOTA: {gs_hota}, DetA: {det_a}, AssA: {ass_a}")
# 手動補正のみ
gs_hota, det_a, ass_a = gshota_man.calc_gs_hota(label_data_list[0], manual_data_list[0], 0)
print(f"手動補正のみ: GS-HOTA: {gs_hota}, DetA: {det_a}, AssA: {ass_a}")
# 自動補正後
gs_hota, det_a, ass_a = gshota_man.calc_gs_hota(label_data_list[0], data_list[0], 0)
print(f"自動補正後: GS-HOTA: {gs_hota}, DetA: {det_a}, AssA: {ass_a}")